# Pair Explanation

Sum attribution score method

1. Extracting too k token pair (sum attribution score)
2. Building related tokens
3. Calculate attribution score


In [ ]:
matcher_names = ["bert_mini", "magellan"]
dataset_root_dir = "../../data/lemon/datasets"
model_root_dir = "../../data/lemon/model"
lime_result_root_dir = "../../data/experiments/10_eval/lime_result"
out_root_dir = "../../data/experiments/11_eval/lime_pair"
dataset_names = [
    "structured_amazon_google",
    "structured_beer",
    "structured_dblp_acm",
    "structured_dblp_google_scholar",
    "structured_fodors_zagat",
    "structured_walmart_amazon",
    "structured_itunes_amazon",
    "dirty_dblp_acm",
    "dirty_dblp_google_scholar",
    "dirty_walmart_amazon",
    "dirty_itunes_amazon",
    "textual_abt_buy",
    "textual_company",
]

In [ ]:
TARGET_DATASET_ID = 1
TOP_N = 5
GPU_ID = 0
LIME_NO_INTERCEPT = False

In [ ]:
BATCH_SIZE = 512
gpu_id = GPU_ID

In [ ]:
# torchモジュールの読み込み前に、利用できるGPUを指定しておく
## これをやらないと、システム内のＧＰＵすべてを利用してしまう
import os

os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_id}"

import torch

print("CUDA =", torch.cuda.is_available())
print("CUDA DEVICES =", torch.cuda.device_count())
print("CUDA CURRENT DEVICE_ID = ", torch.cuda.current_device())

In [ ]:
# transformers の tokenizer を並列実行で呼び出すか（dead lockしてしまう）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Set Random Seeds and Reproducibility
import random

import numpy as np


def set_seed(seed: int):
    """
    Helper function for reproducible behavior to set the seed in ``random``, ``numpy``, ``torch``
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

## データセットを読み込む

In [ ]:
from pine.dataset import load_dataset


def test_load_dataset():
    dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset = load_dataset(dataset_name, dataset_root_dir)
    display(dataset.test.records.a.head())
    display(dataset.test.records.b.head())
    display(dataset.test.record_id_pairs.head())
    display(dataset.test.labels.head())
    print(dataset.test.records.a.dtypes)
    print("Test DATA SIZE =", len(dataset.test.record_id_pairs))


test_load_dataset()

### マッチスコア出力関数

In [ ]:
from pine.matcher.magellan_matcher import make_magellan_matcher_func
from pine.matcher.transformer_matcher import make_transformer_matcher_func
from pine.entity import Entity, EntityPair


def test_proba_fn():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    entity_pairs = []
    for idx in range(5):
        pair_id = dataset.test.record_id_pairs.iloc[idx : idx + 1]
        entity_l = Entity.from_dataframe(
            dataset.test.records.a[
                pair_id.iloc[0].loc["a.rid"] : pair_id.iloc[0]["a.rid"] + 1
            ]
        )
        entity_r = Entity.from_dataframe(
            dataset.test.records.b[
                pair_id.iloc[0].loc["b.rid"] : pair_id.iloc[0]["b.rid"] + 1
            ]
        )
        entity_pairs.append(EntityPair(entity_l, entity_r))

    proba_fn = make_magellan_matcher_func(target_dataset_name, model_root_dir)
    scores = proba_fn(entity_pairs, True)
    for pair, score in zip(entity_pairs, scores):
        display(pair.entity_l.to_dataframe())
        display(pair.entity_r.to_dataframe())
        print(score)


test_proba_fn()

## Lime結果を読み込む

In [ ]:
from typing import Dict, List, Tuple
import pathlib
import pickle
from dataclasses import dataclass
from pine.explainer import AttributionScore


@dataclass
class LimeResult:
    attributions_l: List[AttributionScore]
    attributions_r: List[AttributionScore]
    match_score: float
    lime_intercept: float
    lime_pred_score: float
    lime_match_score: float


def load_lime_result(
    target_dataset_name: str,
    target_matcher_name: str,
    dir_path: pathlib.Path,
) -> Dict[int, Dict[Tuple[int, int], LimeResult]]:
    dir_path = dir_path / target_matcher_name / target_dataset_name
    id_to_pairdel_to_results = {}
    for file_path in dir_path.glob("*.pickle"):
        with file_path.open("rb") as f:
            data = pickle.load(f)

        # lime結果を分かりやすいオブジェクトでラップする
        data_coverted = {}
        for id, result in data.items():
            data_coverted[id] = {}
            for k, v in result.items():
                data_coverted[id][k] = LimeResult(*v)

        id_to_pairdel_to_results.update(data_coverted)
    return id_to_pairdel_to_results


def test_load_lime_result():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    target_matcher_name = matcher_names[0]
    ret = load_lime_result(
        target_dataset_name, target_matcher_name, pathlib.Path(lime_result_root_dir)
    )
    for i, (idx, pair_scores) in enumerate(ret.items()):
        print(idx)
        print(pair_scores)
        if i >= 10:
            break


test_load_lime_result()

# PairToken作成

In [ ]:
from typing import List
from pine.explainer import AttributionScore
from typing import Dict, Tuple, Callable
from dataclasses import dataclass, replace
import copy
import itertools
import lemon

from pine.explainer.lime_explainer import make_explanation, kernel
from pine.entity import EntityPair


@dataclass
class PairSegment:
    index_l: int
    index_r: int
    score: float
    match_score_diff: float
    del_left: bool


def make_pair_segment_list_attribution_score_sum(
    entity_pair: EntityPair,
    lime_results: Dict[Tuple[int, int], LimeResult],
    top_n: int,
    is_match: bool,
) -> List[PairSegment]:
    """対応したセグメントのリストを作成。score順に返す。
    top_nとis_matchに関係なく、すべてのペアの合計を出力する。"""
    pair_segment_list: List[PairSegment] = []

    # 通常Limeの結果
    lime_result_org = lime_results[(None, None)]
    token_attrs_org_l = {x.index: x.score for x in lime_result_org.attributions_l}
    token_attrs_org_r = {x.index: x.score for x in lime_result_org.attributions_r}

    # ペアのAttributuonScore合計を計算する
    attr_score_sum = {}
    for token_idx_l, att_score_l in token_attrs_org_l.items():
        for token_idx_r, att_score_r in token_attrs_org_r.items():
            attr_score_sum[(token_idx_l, token_idx_r)] = att_score_l + att_score_r

    for (token_idx_l, token_idx_r), score in attr_score_sum.items():
        pair_segment_list.append(
            PairSegment(
                token_idx_l,
                token_idx_r,
                score,
                -1,
                True,
            )
        )

    return pair_segment_list


def extract_pair_segments_attribution_score_sum(
    entiry_pair: EntityPair, lime_results: Dict[Tuple[int, int], LimeResult], top_n: int
) -> List[PairSegment]:
    """関連するセグメントIDペアを抽出する

    return
        list: 単語ペアリスト
    """
    pair_segment_list_match: List[PairSegment] = []
    pair_segment_list_unmatch: List[PairSegment] = []
    # もともとのスコアがmatchの場合、マッチ側に対応したセグメントのリストを作成
    #               unmatchの場合、unmatch側に対応したセグメントのリストを作成
    if lime_results[(None, None)].match_score > 0:
        pair_segment_list_match: List[PairSegment] = (
            make_pair_segment_list_attribution_score_sum(
                entiry_pair, lime_results, top_n, True
            )
        )
    else:
        pair_segment_list_unmatch: List[PairSegment] = (
            make_pair_segment_list_attribution_score_sum(
                entiry_pair, lime_results, top_n, False
            )
        )

    # match_score が 正の場合　score の大きいものからtop_n選択
    # match_score が 負の場合　score の小さいものからtop_n選択
    #  ただし、既に選択したtokenを含んでいる場合はSKIP
    pair_segment_list_filtered = []
    already_sel_l = set()
    already_sel_r = set()
    for pair_seg in sorted(
        itertools.chain(pair_segment_list_match, pair_segment_list_unmatch),
        key=lambda x: x.score,
        reverse=True if lime_results[(None, None)].match_score > 0 else False,
    ):
        if len(pair_segment_list_filtered) >= top_n:
            break
        if pair_seg.index_l is not None and pair_seg.index_l in already_sel_l:
            continue
        if pair_seg.index_r is not None and pair_seg.index_r in already_sel_r:
            continue
        already_sel_l.add(pair_seg.index_l)
        already_sel_r.add(pair_seg.index_r)
        pair_segment_list_filtered.append(pair_seg)

    return pair_segment_list_filtered


def test_extract_pair_segments_attribution_score_sum():
    # データセットからmatch score positive 先頭10件、match score negative 先頭10件のデータを選び、結果のペアが想定したものかどうかを計測する
    ## ここでrundomに選択してしまうと、乱数のseedが今までと変わってしまうので、先頭から選択するのみ
    test_data_num = 10
    test_data_pos = {}
    test_data_neg = {}
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    target_matcher_name = matcher_names[0]
    lime_result_dir_path = lime_result_root_dir

    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    lime_results = load_lime_result(
        target_dataset_name,
        target_matcher_name,
        pathlib.Path(lime_result_dir_path),
    )
    for pid, l_id, r_id in dataset.test.record_id_pairs.itertuples():
        pairdel_to_lime_results = lime_results[pid]
        match_score = pairdel_to_lime_results[(None, None)].match_score
        entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
        entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
        entity_pair = EntityPair(entity_l, entity_r)
        if match_score > 0 and len(test_data_pos) <= test_data_num:
            test_data_pos[pid] = (entity_pair, pairdel_to_lime_results)
        elif match_score < 0 and len(test_data_neg) <= test_data_num:
            test_data_neg[pid] = (entity_pair, pairdel_to_lime_results)
        if len(test_data_pos) > test_data_num and len(test_data_neg) > test_data_num:
            break

    for pid, data in itertools.chain(test_data_pos.items(), test_data_neg.items()):
        entity_pair, pairdel_to_lime_results = data
        attr_to_score_l = {
            attr.index: attr.score
            for attr in pairdel_to_lime_results[(None, None)].attributions_l
        }
        attr_to_score_r = {
            attr.index: attr.score
            for attr in pairdel_to_lime_results[(None, None)].attributions_r
        }

        pair_segment_list = extract_pair_segments_attribution_score_sum(
            entity_pair, pairdel_to_lime_results, TOP_N
        )
        # 件数があっている
        assert len(pair_segment_list) <= TOP_N

        # スコアの順番になっている(match_scoreが正の場合、大きい順、負の場合、小さい順)
        if pairdel_to_lime_results[(None, None)].match_score > 0:
            for i in range(len(pair_segment_list) - 1):
                assert pair_segment_list[i].score >= pair_segment_list[i + 1].score
        else:
            for i in range(len(pair_segment_list) - 1):
                assert pair_segment_list[i].score <= pair_segment_list[i + 1].score

        # スコア値があっている
        for pair_seg in pair_segment_list:
            assert pair_seg.score == attr_to_score_l.get(
                pair_seg.index_l, 0
            ) + attr_to_score_r.get(pair_seg.index_r, 0)




# test_normalize_attribution_score()
# test_filter_positive_topk()
test_extract_pair_segments_attribution_score_sum()

In [ ]:
from functools import lru_cache
from typing import Dict, Tuple
import pathlib


@lru_cache()
def load_pair_segment(
    target_dataset_name: str,
    target_matcher_name: str,
    top_n: int,
    dataset_dir_path: pathlib.Path = None,
    lime_result_dir_path: pathlib.Path = None,
) -> Dict[int, Dict[Tuple[int, int], float]]:
    dataset = load_dataset(target_dataset_name, dataset_dir_path)
    lime_results = load_lime_result(
        target_dataset_name, target_matcher_name, lime_result_dir_path
    )
    results = {}
    for pid, l_id, r_id in dataset.test.record_id_pairs.itertuples():
        lime_result = lime_results[pid]
        entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
        entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
        entity_pair = EntityPair(entity_l, entity_r)
        pair_segment_list = extract_pair_segments_attribution_score_sum(entity_pair, lime_result, top_n)
        results[pid] = pair_segment_list
    return results


def test_load_pair_segment():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    target_matcher_name = matcher_names[0]
    results = load_pair_segment(
        target_dataset_name,
        target_matcher_name,
        TOP_N,
        pathlib.Path(dataset_root_dir),
        pathlib.Path(lime_result_root_dir),
    )
    for i, (idx, pair_scores) in enumerate(results.items()):
        print(idx)
        print(pair_scores)
        if i >= 10:
            break


test_load_pair_segment()

# pair expranation 作成

In [ ]:
from pine.explainer.lime_explainer import make_explanation_without_separate_lr
from pine.entity import MergedSegment


@dataclass
class LimeResultPair:
    attributions: List[AttributionScore]
    match_score: float
    lime_intercept: float
    lime_pred_score: float
    lime_match_score: float


def make_pair_explanation(
    entity_pair: EntityPair, pair_segments: List[PairSegment], proba_fn, fit_intercept: bool
) -> Tuple[LimeResultPair, EntityPair]:
    """"""
    merging_segment_list: List[MergedSegment] = []
    for pair_seg in pair_segments:
        merge_seg = MergedSegment([],[])
        if pair_seg.index_l is not None:
            merge_seg.segment_list_in_l.append(pair_seg.index_l)
        if pair_seg.index_r is not None:
            merge_seg.segment_list_in_r.append(pair_seg.index_r)
        merging_segment_list.append(merge_seg)
    entity_pair_merged = entity_pair.make_entity_pair_by_merging_segment_list_only(
        merging_segment_list
    )
    # Explanation対象のペアがない場合は、空のLimeResultPairを返す
    if len(merging_segment_list) == 0:
        return LimeResultPair([], None, None, None, None), entity_pair_merged

    lime_result_pair = LimeResultPair(
        *make_explanation_without_separate_lr(entity_pair_merged, proba_fn, fit_intercept=fit_intercept)
    )
    lime_result_pair.attributions = sorted(
        lime_result_pair.attributions, key=lambda x: abs(x.score), reverse=True
    )
    return lime_result_pair, entity_pair_merged

In [ ]:
import lemon
from tqdm import tqdm


def dump_pair_explanations(
    target_dataset_name: str,
    target_matcher_name: str,
    top_n: int,
    pair_explanation_root_dir_path: pathlib.Path = None,
    dataset_root_dir_path: pathlib.Path = None,
    lime_result_root_dir_path: pathlib.Path = None,
):
    pair_explanations = {}
    pair_explanations_file_path = (
        pair_explanation_root_dir_path
        / target_matcher_name
        / target_dataset_name
        / "pair_explanations.pickle"
    )
    pair_explanations_file_path.parent.mkdir(parents=True, exist_ok=True)

    # すでにデータが保存されいたらロードして終わり
    if pair_explanations_file_path.exists():
        return load_pair_explanations(
            target_dataset_name, target_matcher_name, pair_explanation_root_dir_path
        )

    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    pair_segments = load_pair_segment(
        target_dataset_name,
        target_matcher_name,
        top_n,
        dataset_root_dir_path,
        lime_result_root_dir_path,
    )
    if target_matcher_name == "magellan":
        matcher_func = make_magellan_matcher_func(target_dataset_name, model_root_dir)
    elif target_matcher_name == "bert_mini":
        matcher_func = make_transformer_matcher_func(
            target_dataset_name, model_root_dir
        )

    for pid, l_id, r_id in tqdm(
        dataset.test.record_id_pairs.itertuples(),
        total=len(dataset.test.record_id_pairs),
    ):
        entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
        entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
        entity_pair = EntityPair(entity_l, entity_r)
        segment_pairs = pair_segments[pid]
        if len(segment_pairs) == 0:
            print("WARN:Could not make explanation because No token pair of pid={}.".format(pid))
    
        fit_intercept = not LIME_NO_INTERCEPT
        pair_explanations[pid] = make_pair_explanation(
            entity_pair, segment_pairs, matcher_func, fit_intercept
        )

    with pair_explanations_file_path.open("wb") as f_out:
        pickle.dump(pair_explanations, f_out)
    return pair_explanations


def dump_pair_explanations_all():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    for target_matcher_name in matcher_names:
        dump_pair_explanations(
            target_dataset_name,
            target_matcher_name,
            TOP_N,
            pathlib.Path(out_root_dir),
            pathlib.Path(dataset_root_dir),
            pathlib.Path(lime_result_root_dir),
        )
    return


def load_pair_explanations(
    target_dataset_name: str,
    target_matcher_name: str,
    pair_explanation_root_dir_path: pathlib.Path = None,
) -> Dict[int, Tuple[LimeResultPair, EntityPair]]:
    pair_explanations_file_path = (
        pair_explanation_root_dir_path
        / target_matcher_name
        / target_dataset_name
        / "pair_explanations.pickle"
    )

    with pair_explanations_file_path.open("rb") as f_in:
        pair_explanations = pickle.load(f_in)
    return pair_explanations


def test_load_pair_explanations():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    target_matcher_name = matcher_names[0]
    pair_explanations = load_pair_explanations(
        target_dataset_name, target_matcher_name, pathlib.Path(out_root_dir)
    )
    for i, (idx, exp) in enumerate(pair_explanations.items()):
        print(idx)
        print(exp)
        if i >= 10:
            break


dump_pair_explanations_all()
test_load_pair_explanations()


### 難しいデータを抽出し、何件か出力

- 難しいデータ
  - label = match -> ペアの単語ベースのcos類似度が低い
  - label = unmatch -> ペアの単語ベースのcos類似度が高い 

In [ ]:
from collections import Counter
from math import sqrt
from typing import List

import pandas as pd
import lemon


def cosine_similarity(tokens_a: List[str], tokens_b: List[str]) -> float:
    """テキストのコサイン類似度を計算する"""
    # 各トークン集合の頻度をカウント
    counter_a = Counter(tokens_a)
    counter_b = Counter(tokens_b)

    # 全てのユニークなトークンを取得
    all_tokens = set(counter_a.keys()).union(set(counter_b.keys()))

    # トークンの頻度を基にベクトルを作成
    vec_a = [counter_a.get(token, 0) for token in all_tokens]
    vec_b = [counter_b.get(token, 0) for token in all_tokens]

    # ベクトルのドット積とマグニチュードを計算
    dot_product = sum([a * b for a, b in zip(vec_a, vec_b)])
    magnitude_a = sqrt(sum([a * a for a in vec_a]))
    magnitude_b = sqrt(sum([b * b for b in vec_b]))

    # コサイン類似度を計算
    if magnitude_a * magnitude_b == 0:
        return 0  # 0除算を避ける
    else:
        return dot_product / (magnitude_a * magnitude_b)


def extract_difficult_data(
    dataset: lemon.utils.datasets.Dataset, n: int = 10
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """レコードペアのうち、難しいデータを、n件抽出する。

    label=1のとき
    - cosine類似度が小さいもの
    label=0のとき
    - cosine類似度が大きいもの

    return:
    - match 用の難しいデータ pidリスト
    - Unmatch 用の難しいデータ pidリスト
    """
    a_id_to_text = {}
    b_id_to_text = {}
    scores = []
    record_id_pairs_score = dataset.record_id_pairs.copy()
    for pid, a_id, b_id in dataset.record_id_pairs.itertuples():
        texts_a = a_id_to_text.get(a_id, None)
        texts_b = b_id_to_text.get(b_id, None)
        if texts_a is None:
            entity_a = Entity.from_dataframe(dataset.records.a[a_id : a_id + 1])
            texts_a = [
                entity_a.get_segment_label(idx)
                for idx in range(entity_a.segment_size())
            ]
            a_id_to_text[a_id] = texts_a
        if texts_b is None:
            entity_b = Entity.from_dataframe(dataset.records.b[b_id : b_id + 1])
            texts_b = [
                entity_b.get_segment_label(idx)
                for idx in range(entity_b.segment_size())
            ]
            b_id_to_text[b_id] = texts_b
        cos_sim = cosine_similarity(texts_a, texts_b)
        scores.append(cos_sim)
    record_id_pairs_score["score"] = scores

    # label = True : cos_sim の小さい順 top n
    dataset_pair_match = record_id_pairs_score[dataset.labels == True].sort_values(
        "score", ascending=True
    )[:n]
    # label = false : cos_sim の大きい順 top n
    dataset_pair_unmatch = record_id_pairs_score[dataset.labels == False].sort_values(
        "score", ascending=False
    )[:n]

    return dataset_pair_match, dataset_pair_unmatch


def test_extract_difficult_data():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    match_pair, unmatch_pair = extract_difficult_data(dataset.test, 2)
    display(match_pair)
    for a_id, b_id, _ in match_pair.itertuples(index=False):
        display(dataset.test.records.a.loc[[a_id]])
        display(dataset.test.records.b.loc[[b_id]])
    display(unmatch_pair)
    for a_id, b_id, _ in unmatch_pair.itertuples(index=False):
        display(dataset.test.records.a.loc[[a_id]])
        display(dataset.test.records.b.loc[[b_id]])


test_extract_difficult_data()

In [ ]:
from contextlib import contextmanager

from pine.explainer import AttributionScore


@contextmanager
def option_context(*args, **kwargs):
    original_options = {opt: pd.get_option(opt) for opt in kwargs.keys()}
    pd.set_option(*args, **kwargs)
    try:
        yield
    finally:
        for opt, value in original_options.items():
            pd.set_option(opt, value)


def display_pair_tokens(
    entity_l: Entity,
    entity_r: Entity,
    pair_segments: List[PairSegment],
    topk: int = 5,
):
    for pair_seg in pair_segments[:topk]:
        tokens_l = entity_l.get_segment_label(pair_seg.index_l) if pair_seg.index_l is not None else None
        tokens_r = entity_r.get_segment_label(pair_seg.index_r) if pair_seg.index_r is not None else None
        print(
            tokens_l,
            tokens_r,
            pair_seg.score,
            pair_seg.match_score_diff,
            "del_left" if pair_seg.del_left else "del_right",
        )


def display_pair_explanation(
    pair_explanation: LimeResultPair,
    entity_pair: EntityPair,
):
    for attr_pair in pair_explanation.attributions:
        token_lr = entity_pair.get_segment_label(attr_pair.index)
        print(
            token_lr,
            attr_pair.score,
        )


def display_result(
    entity_l: Entity,
    entity_r: Entity,
    label: int,
    pair_segment: List[PairSegment],
    lime_result: LimeResult,
    topk: int,
    pair_explanation: LimeResultPair,
    pair_explanation_entity_pair: EntityPair,
):
    print("===============================================================")
    with option_context("display.max_colwidth", 200):
        display(entity_l.to_dataframe())
        display(entity_r.to_dataframe())
    print("LABEL =", label)
    print("Match Score =", lime_result.match_score)
    print("======= Related pair tokens")
    display_pair_tokens(entity_l, entity_r, pair_segment, topk)
    display_pair_explanation(pair_explanation, pair_explanation_entity_pair)

In [ ]:
def display_difficult_data(dataset_name: str, matcher_name: str, topk: int = 10):
    dataset = load_dataset(dataset_name, dataset_root_dir)
    lime_results = load_lime_result(
        dataset_name,
        matcher_name,
        pathlib.Path(lime_result_root_dir),
    )
    pair_segments = load_pair_segment(
        dataset_name,
        matcher_name,
        topk,
        pathlib.Path(dataset_root_dir),
        pathlib.Path(lime_result_root_dir),
    )
    pair_explanations = load_pair_explanations(
        dataset_name, matcher_name, pathlib.Path(out_root_dir)
    )
    match_pair_df, unmatch_pair_df = extract_difficult_data(dataset.test, 10)
    print("dataset_name:", dataset_name)
    print("matcher_name:", matcher_name)
    for pid, l_id, r_id, _ in match_pair_df.itertuples():
        entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
        entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
        label = dataset.test.labels.loc[pid]
        lime_result = lime_results[pid][(None, None)]
        pair_segs = pair_segments[pid]
        pair_explanation, pair_explanation_entity_pair = pair_explanations[pid]
        display_result(
            entity_l,
            entity_r,
            label,
            pair_segs,
            lime_result,
            10,
            pair_explanation,
            pair_explanation_entity_pair,
        )
    for pid, l_id, r_id, _ in unmatch_pair_df.itertuples():
        entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
        entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
        label = dataset.test.labels.loc[pid]
        lime_result = lime_results[pid][(None, None)]
        pair_segs = pair_segments[pid]
        pair_explanation, pair_explanation_entity_pair = pair_explanations[pid]
        display_result(
            entity_l,
            entity_r,
            label,
            pair_segs,
            lime_result,
            10,
            pair_explanation,
            pair_explanation_entity_pair,
        )


display_difficult_data(dataset_names[TARGET_DATASET_ID], matcher_names[0], TOP_N)

In [ ]:
display_difficult_data(dataset_names[TARGET_DATASET_ID], matcher_names[1], TOP_N)

### 評価用データ作成

- faithful用
  - dataid -> k -> match_score(k token pair まで追加した時の match score)
- Contributuon用
  - dataid -> k -> match_score(k token pair まで削除した時の match score)
- CosSim用
  - dataid -> k -> cos_sim(k token pair 番目のcos sim)
- ペアで動作しているか
  - dataid -> k 
    - k ペア削除した時のMatchScore
    - kペアの右側を削除した時のMatchScore
    - kペアの左側を削除した時のMatchScore

## failthful用

In [ ]:
from pine.entity import Attribute


def make_match_scores_with_token_pairs(
    entity_pair: EntityPair,
    match_func,
    pair_segments: List[PairSegment],
):
    """pair_segmentsのペアのみを残した場合のスコアリストを返す。
    tokenを残す位置は、元のentity_pairの位置と同じ位置とする。

    pair_segmentsが5個の場合、6個の以下のスコアリストを返す。
    scores[0] = 何も残さない時のスコア
    scores[1] = pair_segments[0]のみ残した場合のスコア
    scores[2] = pair_segments[0]とpair_segments[1]を残した場合のスコア
    scores[3] = pair_segments[0]からpair_segments[2]を残した場合のスコア
    scores[4] = pair_segments[0]からpair_segments[3]を残した場合のスコア
    scores[5] = pair_segments[0]からpair_segments[4]を残した場合のスコア
    """
    entity_pairs = []
    # 何も残さない
    delete_segments = list(range(entity_pair.segment_size()))
    entity_pair_0 = entity_pair.make_entity_pair_by_deleting_segments(delete_segments)
    entity_pairs.append(entity_pair_0)

    for i in range(len(pair_segments)):
        remain_segments_l = [
            pair_token.index_l for pair_token in pair_segments[: i + 1] if pair_token.index_l is not None
        ]
        delete_segments_l = [
            idx
            for idx in range(entity_pair.entity_l.segment_size())
            if idx not in remain_segments_l
        ]
        entity_l = entity_pair.entity_l.make_entity_by_deleting_segments(
            delete_segments_l
        )

        remain_segments_r = [
            pair_token.index_r for pair_token in pair_segments[: i + 1] if pair_token.index_r is not None
        ]
        delete_segments_r = [
            idx
            for idx in range(entity_pair.entity_r.segment_size())
            if idx not in remain_segments_r
        ]
        entity_r = entity_pair.entity_r.make_entity_by_deleting_segments(
            delete_segments_r
        )
        entity_pairs.append(EntityPair(entity_l, entity_r))

    scores = match_func(entity_pairs, False)

    # for i, entity_pair_ in enumerate(entity_pairs):
    #     print(*entity_pair_.to_dataframe())
    #     print(scores[i])

    return scores


def test_make_match_scores_with_token_pairs():
    target_dataset_name = "structured_amazon_google"
    attr_list_1 = [
        Attribute("title", "apple iphone 13", "string"),
        Attribute("manufacturer", "apple", "string"),
        Attribute("price", 10, "Float64"),
    ]
    attr_list_2 = [
        Attribute("title", "apple iphone 12", "string"),
        Attribute("manufacturer", "apple", "string"),
        Attribute("price", 10, "Float64"),
    ]
    entity_pair = EntityPair(Entity(attr_list_1), Entity(attr_list_2))
    func = make_transformer_matcher_func(target_dataset_name, model_root_dir)
    scores = make_match_scores_with_token_pairs(
        entity_pair,
        func,
        [
            PairSegment(0, 0, 0, 0, True),
            PairSegment(0, 1, 0, 0, True),
            PairSegment(1, 1, 0, 0, True),
            PairSegment(3, None, 0, 0, True),
        ],
    )
    print(scores)

    entity_pairs_expected = [
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0, 1, 2, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0, 1, 2, 3]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([1, 2, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([1, 2, 3]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([1, 2, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([2, 3]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([2, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([2, 3]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([2]),
            Entity(attr_list_2).make_entity_by_deleting_segments([2, 3]),
        ),
    ]

    scores_expected = np.array(
        [
            func([entity_pairs_expected[0]], False)[0],
            func([entity_pairs_expected[1]], False)[0],
            func([entity_pairs_expected[2]], False)[0],
            func([entity_pairs_expected[3]], False)[0],
            func([entity_pairs_expected[4]], False)[0],
        ],
        dtype=np.float16,
    )
    np.testing.assert_almost_equal(scores.astype(np.float16), scores_expected)


test_make_match_scores_with_token_pairs()


In [ ]:
from tqdm import tqdm


def eval_match_score_remain(
    dataset: lemon.utils.datasets.SplittedDataset,
    matcher_func: Callable,
    results,
    out_dir_path: pathlib.Path,
):
    score_remains = {}
    score_remains_file_path = out_dir_path / "match_score_remains.pickle"
    if score_remains_file_path.exists():
        # すでに結果があればロードする
        with score_remains_file_path.open("rb") as f_in:
            score_remains = pickle.load(f_in)
    else:
        # 結果ファイルがなければ作成する
        for pid, l_id, r_id in tqdm(
            dataset.test.record_id_pairs.itertuples(),
            total=len(dataset.test.record_id_pairs),
        ):
            entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
            entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
            entity_pair = EntityPair(entity_l, entity_r)
            segment_pairs = results[pid]

            score_remains[pid] = make_match_scores_with_token_pairs(
                entity_pair, matcher_func, segment_pairs
            )

        with score_remains_file_path.open("wb") as f_out:
            pickle.dump(score_remains, f_out)
    return score_remains


In [ ]:
import matplotlib.pyplot as plt


def _calc_post_hoc_acc(
    rank,
    match_scores_pertub,
    lime_results: Dict[int, Dict[Tuple[int, int], LimeResult]],
):
    match_scores_pertub_arr = np.empty((len(match_scores_pertub),), dtype=np.float)
    match_scores_org_arr = np.empty((len(match_scores_pertub),), dtype=np.float)
    for i, pid in enumerate(match_scores_pertub):
        match_scores_pertub_ = match_scores_pertub[pid]
        # 削除できるものがない場合、この前のスコアを採用
        if match_scores_pertub_.size < rank + 1:
            match_scores_pertub_ = np.pad(
                match_scores_pertub[pid],
                (0, rank + 1 - match_scores_pertub_.size),
                "edge",
            )
        # print(match_scores_pertub[pid])
        # print(match_scores_pertub_)
        match_scores_pertub_arr[i] = match_scores_pertub_[rank]
        match_scores_org_arr[i] = lime_results[pid][(None, None)].match_score
    valid_indices = ~np.isnan(match_scores_pertub_arr)
    y = (match_scores_org_arr[valid_indices] > 0).astype(int)
    y_pred = (match_scores_pertub_arr[valid_indices] > 0).astype(int)
    print("len=", len(y_pred), "co=", (y == y_pred).sum())
    return np.mean(y == y_pred)


def display_eval_match_score_remain(target_dataset_name, top_n):
    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    for target_matcher_name in matcher_names:
        if target_matcher_name == "magellan":
            matcher_func = make_magellan_matcher_func(
                target_dataset_name, model_root_dir
            )
        elif target_matcher_name == "bert_mini":
            matcher_func = make_transformer_matcher_func(
                target_dataset_name, model_root_dir
            )
        lime_results = load_lime_result(
            target_dataset_name,
            target_matcher_name,
            pathlib.Path(lime_result_root_dir),
        )
        # results = load_pair_segment(
        #     target_dataset_name,
        #     target_matcher_name,
        #     top_n,
        #     pathlib.Path(dataset_root_dir),
        #     pathlib.Path(lime_result_root_dir),
        # )
        pair_explanations = load_pair_explanations(
            target_dataset_name, target_matcher_name, pathlib.Path(out_root_dir)
        )
        results = {}
        for pid, (pair_ex, pair_ex_entity_pair) in pair_explanations.items():
            pair_segs = []
            # attribution_scoreの絶対値が大きい順にペアを作成
            for attr in sorted(pair_ex.attributions, key=lambda x: abs(x.score), reverse=True):
                segment_list_in_l = pair_ex_entity_pair.merged_segment_list[attr.index].segment_list_in_l
                segment_list_in_r = pair_ex_entity_pair.merged_segment_list[attr.index].segment_list_in_r
                l_idx = segment_list_in_l[0] if len(segment_list_in_l)>0 else None
                r_idx = segment_list_in_r[0] if len(segment_list_in_r)>0 else None
                pair_segs.append(PairSegment(l_idx, r_idx, attr.score, 0, True))
            results[pid] = pair_segs

        out_dir_path = (
            pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
        )
        out_dir_path.mkdir(parents=True, exist_ok=True)

        score_remains = eval_match_score_remain(
            dataset, matcher_func, results, out_dir_path
        )

        post_hoc_accs = []
        for i in range(top_n + 1):
            acc = _calc_post_hoc_acc(i, score_remains, lime_results)
            post_hoc_accs.append(acc)

        print(post_hoc_accs)
        plt.figure(figsize=(15, 6))
        plt.subplot(1, 2, 1)  # 1行2列のグリッドの最初のプロット
        plt.plot(post_hoc_accs, marker="o")
        plt.xlabel("token pairs")
        plt.ylabel("accuracy")
        plt.title("post-hoc accuracy")
        plt.grid(True)
        plt.tight_layout()  # グラフ同士の間隔を調整
        plt.show()


display_eval_match_score_remain(dataset_names[TARGET_DATASET_ID], TOP_N)

## Contribution

In [ ]:
def make_match_scores_with_token_pairs_deleted(
    entity_pair: EntityPair,
    match_func,
    pair_segments: List[PairSegment],
):
    """pair_segmentsのペアを削除した場合のスコアリストを返す。

    pair_segmentsが5個の場合、6個の以下のスコアリストを返す。
    scores[0] = 何も残さない時のスコア
    scores[1] = pair_segments[0]のみ削除した場合のスコア
    scores[2] = pair_segments[0]とpair_segments[1]を削除した場合のスコア
    scores[3] = pair_segments[0]からpair_segments[2]を削除した場合のスコア
    scores[4] = pair_segments[0]からpair_segments[3]を削除した場合のスコア
    scores[5] = pair_segments[0]からpair_segments[4]を削除した場合のスコア
    """
    entity_pairs = []
    # そのまま
    entity_pairs.append(entity_pair)

    for i in range(len(pair_segments)):
        delete_segments_l = [
            pair_token.index_l for pair_token in pair_segments[: i + 1] if pair_token.index_l is not None
        ]
        entity_l = entity_pair.entity_l.make_entity_by_deleting_segments(
            delete_segments_l
        )
        delete_segments_r = [
            pair_token.index_r for pair_token in pair_segments[: i + 1]  if pair_token.index_r is not None
        ]
        entity_r = entity_pair.entity_r.make_entity_by_deleting_segments(
            delete_segments_r
        )
        entity_pairs.append(EntityPair(entity_l, entity_r))

    scores = match_func(entity_pairs, False)

    # for i, entity_pair_ in enumerate(entity_pairs):
    #     print(*entity_pair_.to_dataframe())
    #     print(scores[i])

    return scores


def test_make_match_scores_with_token_pairs_deleted():
    target_dataset_name = "structured_amazon_google"
    attr_list_1 = [
        Attribute("title", "apple iphone 13", "string"),
        Attribute("manufacturer", "apple", "string"),
        Attribute("price", 10, "Float64"),
    ]
    attr_list_2 = [
        Attribute("title", "apple iphone 12", "string"),
        Attribute("manufacturer", "apple", "string"),
        Attribute("price", 10, "Float64"),
    ]
    entity_pair = EntityPair(Entity(attr_list_1), Entity(attr_list_2))
    func = make_transformer_matcher_func(target_dataset_name, model_root_dir)
    scores = make_match_scores_with_token_pairs_deleted(
        entity_pair,
        func,
        [
            PairSegment(0, 0, 0, 0, True),
            PairSegment(0, 1, 0, 0, True),
            PairSegment(1, 1, 0, 0, True),
            PairSegment(3, None, 0, 0, True),
        ],
    )
    print(scores)

    entity_pairs_expected = [
        EntityPair(
            Entity(attr_list_1),
            Entity(attr_list_2),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0, 1]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0, 1]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0, 1]),
        ),
        EntityPair(
            Entity(attr_list_1).make_entity_by_deleting_segments([0, 1, 3]),
            Entity(attr_list_2).make_entity_by_deleting_segments([0, 1]),
        ),
    ]

    scores_expected = np.array(
        [
            func([entity_pairs_expected[0]], False)[0],
            func([entity_pairs_expected[1]], False)[0],
            func([entity_pairs_expected[2]], False)[0],
            func([entity_pairs_expected[3]], False)[0],
            func([entity_pairs_expected[4]], False)[0],
        ],
        dtype=np.float16,
    )
    np.testing.assert_almost_equal(scores.astype(np.float16), scores_expected)


test_make_match_scores_with_token_pairs_deleted()

In [ ]:
def dump_match_score_delete(
    target_dataset_name: str, target_matcher_name: str, out_root_dir: pathlib.Path
):
    score_deletes = {}
    out_dir_path = (
        pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
    )
    out_dir_path.mkdir(parents=True, exist_ok=True)
    score_deletes_file_path = out_dir_path / "match_score_deletes.pickle"
    if score_deletes_file_path.exists():
        # すでに結果があればロードする
        with score_deletes_file_path.open("rb") as f_in:
            score_deletes = pickle.load(f_in)
    else:
        # 結果ファイルがなければ作成する
        dataset = load_dataset(target_dataset_name, dataset_root_dir)
        if target_matcher_name == "magellan":
            matcher_func = make_magellan_matcher_func(
                target_dataset_name, model_root_dir
            )
        elif target_matcher_name == "bert_mini":
            matcher_func = make_transformer_matcher_func(
                target_dataset_name, model_root_dir
            )
        pair_explanations = load_pair_explanations(
            target_dataset_name, target_matcher_name, pathlib.Path(out_root_dir)
        )

        for pid, l_id, r_id in tqdm(
            dataset.test.record_id_pairs.itertuples(),
            total=len(dataset.test.record_id_pairs),
        ):
            entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
            entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
            entity_pair = EntityPair(entity_l, entity_r)

            segment_pairs = []
            pair_ex, pair_ex_entity_pair = pair_explanations[pid]
            for attr in pair_ex.attributions:
                segment_list_in_l = pair_ex_entity_pair.merged_segment_list[attr.index].segment_list_in_l
                segment_list_in_r = pair_ex_entity_pair.merged_segment_list[attr.index].segment_list_in_r
                l_idx = segment_list_in_l[0] if len(segment_list_in_l)>0 else None
                r_idx = segment_list_in_r[0] if len(segment_list_in_r)>0 else None
                segment_pairs.append(PairSegment(l_idx, r_idx, attr.score, 0, True))

            if dataset.test.labels.loc[pid]:
                # match のentity pair の場合、match に有効なものから順番（attr.scoreの降順）に削除対象
                segment_pairs = sorted(segment_pairs, key=lambda x:x.score, reverse=True)
            else:
                # unmatch のentity pair の場合、unmatch に有効なものから順番（attr.scoreの昇順）に削除対象
                segment_pairs = sorted(segment_pairs, key=lambda x:x.score)
            
            score_deletes[pid] = make_match_scores_with_token_pairs_deleted(
                entity_pair, matcher_func, segment_pairs
            )

        with score_deletes_file_path.open("wb") as f_out:
            pickle.dump(score_deletes, f_out)
    return score_deletes


def load_dump_match_score_delete(
    target_dataset_name: str, target_matcher_name: str, out_root_dir: pathlib.Path
):
    out_dir_path = (
        pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
    )
    score_deletes_file_path = out_dir_path / "match_score_deletes.pickle"
    with score_deletes_file_path.open("rb") as f_in:
        score_deletes = pickle.load(f_in)
    return score_deletes


In [ ]:
def _calc_posthoc_f1(
    rank, match_scores_pertub: Dict[int, np.array], correct: pd.Series
):
    match_scores_pertub_arr = np.empty((len(match_scores_pertub),), dtype=np.float)
    y = correct.loc[match_scores_pertub.keys()].to_numpy()
    for i, pid in enumerate(match_scores_pertub):
        match_scores_pertub_ = match_scores_pertub[pid]
        # 削除できるものがない場合、この前のスコアを採用
        if match_scores_pertub_.size < rank + 1:
            match_scores_pertub_ = np.pad(
                match_scores_pertub[pid],
                (0, rank + 1 - match_scores_pertub_.size),
                "edge",
            )
        # print(match_scores_pertub[pid])
        # print(match_scores_pertub_)
        match_scores_pertub_arr[i] = match_scores_pertub_[rank]
    valid_indices = ~np.isnan(match_scores_pertub_arr)
    y = y[valid_indices].astype(int)
    y_pred = (match_scores_pertub_arr[valid_indices] > 0).astype(int)
    tp = np.sum((y == 1) & (y_pred == 1))
    fp = np.sum((y == 0) & (y_pred == 1))
    fn = np.sum((y == 1) & (y_pred == 0))
    print("len=", len(y), "tp=", tp, "fp=", fp, "fn=", fn)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        (2 * precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )
    return f1


def display_eval_match_score_delete(target_dataset_name, top_n):
    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    for target_matcher_name in matcher_names:
        score_deletes = dump_match_score_delete(
            target_dataset_name, target_matcher_name, pathlib.Path(out_root_dir)
        )

        post_hoc_f1 = []
        for i in range(top_n + 1):
            f1 = _calc_posthoc_f1(i, score_deletes, dataset.test.labels)
            post_hoc_f1.append(f1)

        print(post_hoc_f1)
        plt.figure(figsize=(15, 6))
        plt.subplot(1, 2, 1)  # 1行2列のグリッドの最初のプロット
        plt.plot(post_hoc_f1, marker="o")
        plt.xlabel("token pairs")
        plt.ylabel("f1")
        plt.title("post-hoc f1")
        plt.grid(True)
        plt.tight_layout()  # グラフ同士の間隔を調整
        plt.show()


display_eval_match_score_delete(dataset_names[TARGET_DATASET_ID], TOP_N)

### ペアの単語が類似しているか計測

- BERTでペアの単語cosine類似度を計測
- ペア単語のcosine類似度を累積

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from scipy.spatial.distance import cosine

device = "cuda" if torch.cuda.is_available() else "cpu"

# モデルとトークナイザーの準備
bert_model_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name).to(device)

In [ ]:
def calculate_cosine_similarities_with_mean_pooling(word_pairs)->np.array:
    # 0件なら0件で返す
    if len(word_pairs) == 0:
        return np.array([])

    word1_list = [pair[0] for pair in word_pairs]
    word2_list = [pair[1] for pair in word_pairs]

    inputs1 = bert_tokenizer(
        word1_list, return_tensors="pt", truncation=True, padding=True, max_length=128
    ).to(bert_model.device)
    inputs2 = bert_tokenizer(
        word2_list, return_tensors="pt", truncation=True, padding=True, max_length=128
    ).to(bert_model.device)

    with torch.no_grad():
        outputs1 = bert_model(**inputs1)
        outputs2 = bert_model(**inputs2)

    # 平均Poolingを行う
    mask1 = (
        inputs1["attention_mask"]
        .unsqueeze(-1)
        .expand_as(outputs1.last_hidden_state)
        .float()
    )
    mask2 = (
        inputs2["attention_mask"]
        .unsqueeze(-1)
        .expand_as(outputs2.last_hidden_state)
        .float()
    )

    embed1 = torch.sum(outputs1.last_hidden_state * mask1, 1) / mask1.sum(1)
    embed2 = torch.sum(outputs2.last_hidden_state * mask2, 1) / mask2.sum(1)

    # コサイン類似度を計算し、numpy配列として返す
    similarities = (
        torch.nn.functional.cosine_similarity(embed1, embed2).to("cpu").numpy()
    )
    return similarities


def calc_segment_pair_cos_sim_each(
    entity_pair: EntityPair,
    pair_segments: List[PairSegment],
)->np.array:
    word_pairs = []
    no_pair_idx = []
    for idx, pair_seg in enumerate(pair_segments):
        if pair_seg.index_l is not None and pair_seg.index_r is not None:
            word_l = entity_pair.entity_l.get_segment_label(pair_seg.index_l)
            word_r = entity_pair.entity_r.get_segment_label(pair_seg.index_r)
            word_pairs.append((word_l, word_r))
        else:
            no_pair_idx.append(idx)

    scores = calculate_cosine_similarities_with_mean_pooling(word_pairs)
    # どちらかがnoneだった場合計算できないので、Noneを挿入する
    result = [None] * len(pair_segments)
    valid_idx = 0

    for i in range(len(pair_segments)):
        if i in no_pair_idx:
            result[i] = None
        else:
            result[i] = scores[valid_idx]
            valid_idx += 1

    return np.array(result)


def test_calculate_cosine_similarities_with_mean_pooling():
    # 類似度の計算
    word_pairs = [
        ("dog", "dog"),
        ("dog", "cat"),
        ("computer", "program"),
        ("book", "page"),
    ]
    similarities = calculate_cosine_similarities_with_mean_pooling(word_pairs)

    for (word1, word2), sim in zip(word_pairs, similarities):
        print(f"コサイン類似度 between '{word1}' and '{word2}': {sim}")

test_calculate_cosine_similarities_with_mean_pooling()

In [ ]:
def eval_pair_cosine(
    dataset: lemon.utils.datasets.SplittedDataset,
    results,
    out_dir_path: pathlib.Path,
):
    """各ペアのコサイン類似度をBERTで計測"""
    cosine_sims_each = {}
    cosine_sims_each_file_path = out_dir_path / "cosine_sims_each.pickle"
    if cosine_sims_each_file_path.exists():
        with cosine_sims_each_file_path.open("rb") as f_in:
            cosine_sims_each = pickle.load(f_in)
    else:
        # 途中結果ファイルがなければ作成する
        for pid, l_id, r_id in tqdm(
            dataset.test.record_id_pairs.itertuples(),
            total=len(dataset.test.record_id_pairs),
        ):
            entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
            entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
            entity_pair = EntityPair(entity_l, entity_r)
            result_data = results[pid]

            cosine_sims_each[pid] = calc_segment_pair_cos_sim_each(
                entity_pair, result_data
            )

        with cosine_sims_each_file_path.open("wb") as f_out:
            pickle.dump(cosine_sims_each, f_out)
    return cosine_sims_each

In [ ]:
def _get_average(
    scores: List[np.ndarray],
    max_length: int = None,
    mode: str = "edge",
    dump: bool = False,
) -> np.ndarray:
    if max_length is None:
        max_length = max([len(s) for s in scores])
    if mode == "constant":
        scores = [
            np.pad(
                s[:max_length].astype(float),
                (0, max_length - s[:max_length].shape[0]),
                mode="constant",
                constant_values=np.nan,
            )
            if len(s) != 0
            else np.array([np.nan] * max_length)
            for s in scores
        ]
    else:
        scores = [
            np.pad(
                s[:max_length].astype(float),
                (0, max_length - s[:max_length].shape[0]),
                mode=mode,
            )
            if len(s) != 0
            else np.array([np.nan] * max_length)
            for s in scores
        ]
    if dump:
        print(scores)
    if len(scores) == 0:
        return np.zeros((max_length,))
    scores = np.vstack(scores)
    return np.nanmean(scores, axis=0)


def cumsum_with_none(arr):
    # None を 0 に置き換え
    arr = np.where(arr == None, 0, arr).astype(np.float64)
    # 累積和を計算
    return np.cumsum(arr)


def display_eval_pair_cosine(target_dataset_name: str, top_n: int):
    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    for target_matcher_name in matcher_names:
        lime_results = load_lime_result(
            target_dataset_name,
            target_matcher_name,
            pathlib.Path(lime_result_root_dir),
        )
        # results = load_pair_segment(
        #     target_dataset_name,
        #     target_matcher_name,
        #     top_n,
        #     pathlib.Path(dataset_root_dir),
        #     pathlib.Path(lime_result_root_dir),
        # )
        pair_explanations = load_pair_explanations(
            target_dataset_name, target_matcher_name, pathlib.Path(out_root_dir)
        )
        results = {}
        for pid, (pair_ex, pair_ex_entity_pair) in pair_explanations.items():
            pair_segs = []
            for attr in pair_ex.attributions:
                segment_list_in_l = pair_ex_entity_pair.merged_segment_list[attr.index].segment_list_in_l
                segment_list_in_r = pair_ex_entity_pair.merged_segment_list[attr.index].segment_list_in_r
                l_idx = segment_list_in_l[0] if len(segment_list_in_l)>0 else None
                r_idx = segment_list_in_r[0] if len(segment_list_in_r)>0 else None
                pair_segs.append(PairSegment(l_idx, r_idx, attr.score, 0, True))
            results[pid] = pair_segs

        out_dir_path = (
            pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
        )
        out_dir_path.mkdir(parents=True, exist_ok=True)

        cos_sims_each = eval_pair_cosine(dataset, results, out_dir_path)

        # match の entity pair と、unmatch の entity pairを分ける
        cos_sims_each_match = {}
        cos_sims_each_unmatch = {}
        for pid, lime_result in lime_results.items():
            match_score_org = lime_result[(None, None)].match_score
            if match_score_org > 0:
                cos_sims_each_match[pid] = cos_sims_each[pid]
            else:
                cos_sims_each_unmatch[pid] = cos_sims_each[pid]

        # コサイン値の積み上げの平均
        cos_sims_match_averaged = _get_average([cumsum_with_none(arr) for arr in cos_sims_each_match.values()])
        cos_sims_unmatch_averaged = _get_average([cumsum_with_none(arr) for arr in cos_sims_each_unmatch.values()])
        if len(cos_sims_match_averaged) <= top_n:
            no_data_length = top_n - len(cos_sims_match_averaged)
            cos_sims_match_averaged = np.pad(
                cos_sims_match_averaged,
                (0, no_data_length),
                mode="constant",
                constant_values=np.nan,
            )
        if len(cos_sims_unmatch_averaged) <= top_n:
            no_data_length = top_n - len(cos_sims_unmatch_averaged)
            cos_sims_unmatch_averaged = np.pad(
                cos_sims_unmatch_averaged,
                (0, no_data_length),
                mode="constant",
                constant_values=np.nan,
            )

        plt.figure(figsize=(15, 6))
        plt.subplot(1, 2, 1)  # 1行2列のグリッドの最初のプロット
        plt.plot(range(1, top_n + 1), cos_sims_match_averaged, marker="o")
        plt.xlim(1, 5)
        plt.xlabel("Pair tokens")
        plt.ylabel("cos sim")
        plt.title("Match Pair tokens")
        plt.grid(True)
        plt.subplot(1, 2, 2)  # 1行2列のグリッドの二つ目のプロット
        plt.plot(range(1, top_n + 1), cos_sims_unmatch_averaged, marker="o")
        plt.xlim(1, 5)
        plt.xlabel("Pair tokens")
        plt.ylabel("cos sim")
        plt.title("UnMatch Pair tokens")
        plt.grid(True)
        plt.tight_layout()  # グラフ同士の間隔を調整
        plt.show()
    return


display_eval_pair_cosine(dataset_names[TARGET_DATASET_ID], TOP_N)

### 考察用データ出力
- Entity_L, Entity_R, is_match, match_scoretoken_L_1, org_score, token_R_1, org_score, token_L_2, org_score, token_R_2, org_score, token_L_3, org_score, token_R_3, org_score, token_L_4, org_score, token_R_4, org_score, token_L_5, org_score, token_R_5, org_score

In [ ]:
import pandas as pd
from typing import Any


def load_pair_tokens_del_result(
    dataset_name: str, matcher_name: str, data_root_dir: str
) -> Dict[Any, np.ndarray]:
    data_dir_path = pathlib.Path(data_root_dir) / matcher_name / dataset_name
    with (data_dir_path / "score_deletions_match.pickle").open("rb") as f:
        score_deletions_match = pickle.load(f)
    with (data_dir_path / "score_deletions_unmatch.pickle").open("rb") as f:
        score_deletions_unmatch = pickle.load(f)
    return score_deletions_match, score_deletions_unmatch


def record_to_prefixed_dict(df, index, prefix):
    """
    指定された index の DataFrame のレコードを辞書に変換し、
    カラム名に指定された接頭辞を追加します。

    Parameters:
    - df: pd.DataFrame, 変換する DataFrame
    - index: int, 変換するレコードの index
    - prefix: str, カラム名に追加する接頭辞

    Returns:
    - dict: 指定されたレコードの辞書表現。キーは接頭辞を追加したカラム名です。
    """
    # 指定された index のレコードを取得
    record = df.loc[index]

    # カラム名に接頭辞を追加して辞書を作成
    prefixed_dict = {f"{prefix}_{key}": value for key, value in record.items()}

    return prefixed_dict


def make_dump_data(
    dataset: lemon.utils.datasets.SplittedDataset,
    lime_result: Dict[int, Dict[Tuple[int, int], LimeResult]],
    pair_explanations: Dict[int, Tuple[LimeResultPair, EntityPair]],
    top_n: int,
    score_deletes: Dict[int, np.ndarray],
):
    """ダンプデータを作成する"""
    dump_data = []
    for pid, l_id, r_id in tqdm(
        dataset.test.record_id_pairs.itertuples(),
        total=len(dataset.test.record_id_pairs),
    ):
        lime_result_org = lime_result[pid][(None, None)]
        lime_result_pair, entity_pair_on_pair_explanation = pair_explanations[pid]
        score_delete = score_deletes[pid]

        entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
        entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])

        data = {}
        data["Pair_ID"] = pid
        data.update(record_to_prefixed_dict(dataset.test.records.a, l_id, "Entity_L"))
        data.update(record_to_prefixed_dict(dataset.test.records.b, r_id, "Entity_R"))
        data["is_match"] = dataset.test.labels.loc[pid]
        data["match_score"] = lime_result_org.match_score

        attributions_l_org = { x.index: x.score for x in lime_result_org.attributions_l}
        attributions_r_org = { x.index: x.score for x in lime_result_org.attributions_r}
        for i, attr in enumerate(lime_result_pair.attributions[:top_n]):
            segment_list_in_l = entity_pair_on_pair_explanation.merged_segment_list[attr.index].segment_list_in_l
            segment_list_in_r = entity_pair_on_pair_explanation.merged_segment_list[attr.index].segment_list_in_r
            index_l = segment_list_in_l[0] if len(segment_list_in_l)>0 else None
            index_r = segment_list_in_r[0] if len(segment_list_in_r)>0 else None
            data[f"token_l_{i}"] = entity_l.get_segment_label(index_l) if index_l is not None else None
            data[f"token_r_{i}"] = entity_r.get_segment_label(index_r) if index_r is not None else None
            data[f"att_score_pair_{i}"] = attr.score
            data[f"att_score_org_l_{i}"] = attributions_l_org[index_l] if index_l is not None else None
            data[f"att_score_org_r_{i}"] = attributions_r_org[index_r] if index_r is not None else None
            data[f"del_match_score_{i}"] = float(score_delete[i + 1])
        dump_data.append(data)

    return dump_data


def dump_pairs(target_dataset_name: str, top_n: int):
    dataset = load_dataset(target_dataset_name, dataset_root_dir)
    for target_matcher_name in matcher_names:
        lime_results = load_lime_result(
            target_dataset_name,
            target_matcher_name,
            pathlib.Path(lime_result_root_dir),
        )
        results = load_pair_explanations(target_dataset_name, target_matcher_name, pathlib.Path(out_root_dir))
        score_deletions = load_dump_match_score_delete(target_dataset_name, target_matcher_name, pathlib.Path(out_root_dir))
        out_dir_path = (
            pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
        )
        out_dir_path.mkdir(parents=True, exist_ok=True)

        dump_data = make_dump_data(
            dataset, lime_results, results, top_n, score_deletions
        )
        pd.DataFrame(dump_data).to_csv(
            out_dir_path / "dump_data.csv", sep="\t", index=False
        )


dump_pairs(dataset_names[TARGET_DATASET_ID], TOP_N)